# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


# MLB 종결 모델 — Daily Predictor
**사용법:** `TARGET_DATE`만 바꾸고 전체 실행 (Ctrl+F9)

- 승패 선택: 확신 64%+ (OOS 74.1%)
- 총득점 선택: 확신 60%+ (OOS 72.3%)

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# ★ 여기만 바꾸면 됨
# ============================================================
TARGET_DATE = "2026-04-01"

MLB_PATH = "<DRIVE_ROOT>/MLB데이터"
DATA_2026 = f"{MLB_PATH}/2026"
OL_PATH = f"{MLB_PATH}/MARKET_PRICE,LINEUP"

st = pd.read_csv(f"{OL_PATH}/mlb_starters_2026.csv")
market_price = pd.read_csv(f"{OL_PATH}/mlb_market_price_2026.csv")
lineup = pd.read_csv(f"{OL_PATH}/mlb_lineup_2026.csv")
results = pd.read_csv(f"{DATA_2026}/mlb_2026_정규시즌_경기결과.csv")
pit = pd.read_csv(f"{DATA_2026}/mlb_2026_정규시즌_투수개인기록.csv")
pf = pd.read_csv(f"{MLB_PATH}/park_factors_2025.csv")

ump_prev = pd.concat([pd.read_csv(f"{MLB_PATH}/{yr}/mlb_{yr}_심판.csv") for yr in [2023,2024,2025]], ignore_index=True)
ump_prev = ump_prev[ump_prev["포지션"]=="Home Plate"]
master_prev = pd.read_csv(f"{MLB_PATH}/B안_마스터테이블.csv")
ump_prev = ump_prev.merge(master_prev[["gamePk","총득점"]].drop_duplicates(), on="gamePk", how="inner")
ump_2026 = pd.read_csv(f"{DATA_2026}/mlb_2026_심판.csv")
ump_2026_hp = ump_2026[ump_2026["포지션"]=="Home Plate"][["gamePk","심판명"]].drop_duplicates()

win_lr = pickle.load(open(f"{MLB_PATH}/종결_승패_LR.pkl","rb"))
win_lgb = pickle.load(open(f"{MLB_PATH}/종결_승패_LGB.pkl","rb"))
run_total_lr = pickle.load(open(f"{MLB_PATH}/종결_총득점_LR.pkl","rb"))
run_total_lgb = pickle.load(open(f"{MLB_PATH}/종결_총득점_LGB.pkl","rb"))

print(f"✅ 데이터 로드 완료 — {TARGET_DATE}")

In [ ]:
WIN_FEATS = ["era_diff","k9_diff","bb9_diff","ops_diff","wr5_diff","wr10_diff",
             "rs10_diff","bp_fatigue_1d_diff","run_total_baseline","run_margin","home_WIN_LOSS",
             "park_HR","ump_avg_runs","ump_vs_league","temperature"]
RUN_TOTAL_FEATS = ["era_sum","whip_sum","k9_sum","bb9_sum","hr9_sum","ops_sum",
            "rs5_sum","rs10_sum","ra5_sum","ra10_sum","bp_fatigue_sum",
            "park_factor","park_R","park_HR","park_H","temperature",
            "ump_avg_runs","ump_vs_league","era_max","whip_max",
            "run_margin","home_WIN_LOSS","run_total_baseline","era_vs_run_total","ops_vs_run_total"]

TEAM_NORMALIZE = {"Oakland Athletics":"Athletics"}
PF_MAP = {
    "Rockies":"Colorado Rockies","Athletics":"Athletics","Tigers":"Detroit Tigers",
    "Reds":"Cincinnati Reds","Orioles":"Baltimore Orioles","Rangers":"Texas Rangers",
    "Red Sox":"Boston Red Sox","Cubs":"Chicago Cubs","Twins":"Minnesota Twins",
    "Brewers":"Milwaukee Brewers","Phillies":"Philadelphia Phillies","Yankees":"New York Yankees",
    "Braves":"Atlanta Braves","Cardinals":"St. Louis Cardinals","Astros":"Houston Astros",
    "Blue Jays":"Toronto Blue Jays","Mets":"New York Mets","Royals":"Kansas City Royals",
    "Diamondbacks":"Arizona Diamondbacks","Nationals":"Washington Nationals",
    "Dodgers":"Los Angeles Dodgers","Pirates":"Pittsburgh Pirates","Padres":"San Diego Padres",
    "Giants":"San Francisco Giants","Marlins":"Miami Marlins","Angels":"Los Angeles Angels",
    "Guardians":"Cleveland Guardians","White Sox":"Chicago White Sox",
    "Rays":"Tampa Bay Rays","Mariners":"Seattle Mariners"
}
print("✅ 상수 설정 완료")

In [ ]:
# ============================================================
# 예측 실행
# ============================================================
day_st = st[st["date"]==TARGET_DATE].copy()
if len(day_st)==0:
    print(f"⚠️ {TARGET_DATE} starters 없음")
else:
    base = day_st[["gamePk","date","away_team","home_team",
        "home_sp_name","home_sp_era","home_sp_whip","home_sp_k9","home_sp_bb9","home_sp_hr9",
        "away_sp_name","away_sp_era","away_sp_whip","away_sp_k9","away_sp_bb9","away_sp_hr9"]].copy()

    # market_price + lineup + 파크팩터 + 경기결과 + 심판
    base = base.merge(market_price[["date","home_team","home_WIN_LOSS","away_WIN_LOSS","run_total_baseline","run_margin","temperature"]],
                      on=["date","home_team"], how="left")
    lu_ops = lineup.groupby(["gamePk","side"])["ops"].mean().reset_index()
    base = base.merge(lu_ops[lu_ops["side"]=="home"][["gamePk","ops"]].rename(columns={"ops":"home_lineup_ops"}), on="gamePk", how="left")
    base = base.merge(lu_ops[lu_ops["side"]=="away"][["gamePk","ops"]].rename(columns={"ops":"away_lineup_ops"}), on="gamePk", how="left")
    pf_tmp = pf.copy()
    pf_tmp["home_team"] = pf_tmp["Team"].map(PF_MAP)
    base = base.merge(pf_tmp[["home_team","ParkFactor","R","HR","H"]].rename(
        columns={"ParkFactor":"park_factor","R":"park_R","HR":"park_HR","H":"park_H"}), on="home_team", how="left")
    base = base.merge(results[["gamePk","홈팀득점","원정팀득점","홈팀결과","총득점"]].drop_duplicates(), on="gamePk", how="left")
    base = base.merge(ump_2026_hp, on="gamePk", how="left")

    # === rolling 피처 ===
    target_dt = pd.Timestamp(TARGET_DATE)
    prev_results = results[pd.to_datetime(results["날짜"]) < target_dt]

    # 최근폼
    form = {}
    if len(prev_results) > 0:
        prev_sorted = prev_results.sort_values("날짜")
        for team in set(prev_sorted["홈팀"]) | set(prev_sorted["원정팀"]):
            tg = []
            for _, r in prev_sorted[(prev_sorted["홈팀"]==team)|(prev_sorted["원정팀"]==team)].iterrows():
                if r["홈팀"]==team:
                    tg.append((1 if r["홈팀결과"]=="승" else 0, r["홈팀득점"], r["원정팀득점"]))
                else:
                    tg.append((1 if r["원정팀결과"]=="승" else 0, r["원정팀득점"], r["홈팀득점"]))
            form[team] = {}
            for w, s in [(5,"5"),(10,"10")]:
                rec = tg[-w:]
                if len(rec) >= 3:
                    form[team][f"wr{s}"] = np.mean([g[0] for g in rec])
                    form[team][f"rs{s}"] = np.mean([g[1] for g in rec])
                    form[team][f"ra{s}"] = np.mean([g[2] for g in rec])

    for idx, row in base.iterrows():
        for tc, pf_ in [("home_team","home"),("away_team","away")]:
            tf = form.get(row[tc], {})
            for k in ["wr5","wr10","rs5","rs10","ra5","ra10"]:
                base.loc[idx, f"{pf_}_{k}"] = tf.get(k, np.nan)

    # 불펜소모도
    starter_set = set()
    for _, r in st.iterrows():
        starter_set.add((r["gamePk"], r.get("home_sp_name","")))
        starter_set.add((r["gamePk"], r.get("away_sp_name","")))
    pit_c = pit.copy()
    pit_c["is_starter"] = pit_c.apply(lambda r: (r["gamePk"], r["선수명"]) in starter_set, axis=1)
    bp = pit_c[~pit_c["is_starter"]]
    bp_game = bp.groupby(["gamePk","팀"]).agg(bp_ip=("이닝","sum")).reset_index()
    bp_game["팀"] = bp_game["팀"].apply(lambda x: TEAM_NORMALIZE.get(x,x))
    # base 팀명도 동일하게 normalize (bp lookup용 임시 컬럼)
    base["_home_norm"] = base["home_team"].map(lambda x: TEAM_NORMALIZE.get(x,x))
    base["_away_norm"] = base["away_team"].map(lambda x: TEAM_NORMALIZE.get(x,x))
    bp_game = bp_game.merge(results[["gamePk","날짜"]].drop_duplicates(), on="gamePk", how="left")
    bp_game["날짜"] = pd.to_datetime(bp_game["날짜"])
    d1 = (target_dt - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    d3 = (target_dt - pd.Timedelta(days=3)).strftime("%Y-%m-%d")
    prev_bp = bp_game[bp_game["날짜"] < target_dt]

    for idx, row in base.iterrows():
        for nc, pf_ in [("_home_norm","home"),("_away_norm","away")]:
            tbp = prev_bp[prev_bp["팀"]==row[nc]]
            base.loc[idx, f"{pf_}_bp_fatigue_1d"] = tbp[tbp["날짜"]>=d1]["bp_ip"].sum()
            base.loc[idx, f"{pf_}_bp_fatigue_3d"] = tbp[tbp["날짜"]>=d3]["bp_ip"].sum()

    # 심판 (교차누적)
    ump_hist = {}
    all_runs = []
    for _, r in ump_prev.iterrows():
        ump_hist.setdefault(r["심판명"], []).append(r["총득점"])
        all_runs.append(r["총득점"])
    prev_ump = ump_2026_hp[ump_2026_hp["gamePk"].isin(set(prev_results["gamePk"]))]
    for _, r in prev_ump.merge(results[["gamePk","총득점"]].drop_duplicates(), on="gamePk").iterrows():
        ump_hist.setdefault(r["심판명"], []).append(r["총득점"])
        all_runs.append(r["총득점"])
    league_avg = np.mean(all_runs)

    for idx, row in base.iterrows():
        ump = row.get("심판명","")
        hist = ump_hist.get(ump, [])
        if len(hist) >= 5:
            base.loc[idx, "ump_avg_runs"] = np.mean(hist)
            base.loc[idx, "ump_vs_league"] = np.mean(hist) - league_avg

    # 임시 컬럼 정리
    base.drop(columns=["_home_norm","_away_norm"], inplace=True)

    # === 파생 피처 ===
    base["era_diff"] = base["home_sp_era"] - base["away_sp_era"]
    base["k9_diff"] = base["home_sp_k9"] - base["away_sp_k9"]
    base["bb9_diff"] = base["home_sp_bb9"] - base["away_sp_bb9"]
    base["ops_diff"] = base["home_lineup_ops"] - base["away_lineup_ops"]
    base["wr5_diff"] = base["home_wr5"] - base["away_wr5"]
    base["wr10_diff"] = base["home_wr10"] - base["away_wr10"]
    base["rs10_diff"] = base["home_rs10"] - base["away_rs10"]
    base["bp_fatigue_1d_diff"] = base["home_bp_fatigue_1d"] - base["away_bp_fatigue_1d"]
    base["era_sum"] = base["home_sp_era"] + base["away_sp_era"]
    base["whip_sum"] = base["home_sp_whip"] + base["away_sp_whip"]
    base["k9_sum"] = base["home_sp_k9"] + base["away_sp_k9"]
    base["bb9_sum"] = base["home_sp_bb9"] + base["away_sp_bb9"]
    base["hr9_sum"] = base["home_sp_hr9"] + base["away_sp_hr9"]
    base["ops_sum"] = base["home_lineup_ops"] + base["away_lineup_ops"]
    base["rs5_sum"] = base["home_rs5"] + base["away_rs5"]
    base["rs10_sum"] = base["home_rs10"] + base["away_rs10"]
    base["ra5_sum"] = base["home_ra5"] + base["away_ra5"]
    base["ra10_sum"] = base["home_ra10"] + base["away_ra10"]
    base["bp_fatigue_sum"] = base["home_bp_fatigue_3d"] + base["away_bp_fatigue_3d"]
    base["era_max"] = base[["home_sp_era","away_sp_era"]].max(axis=1)
    base["whip_max"] = base[["home_sp_whip","away_sp_whip"]].max(axis=1)
    base["era_vs_run_total"] = base["era_sum"] - base["run_total_baseline"]
    base["ops_vs_run_total"] = base["ops_sum"] * 10 - base["run_total_baseline"]

    # === 예측 ===
    wp = 0.8*win_lr.predict_proba(base[WIN_FEATS])[:,1] + 0.2*win_lgb.predict_proba(base[WIN_FEATS])[:,1]
    op = 0.5*run_total_lr.predict_proba(base[RUN_TOTAL_FEATS])[:,1] + 0.5*run_total_lgb.predict_proba(base[RUN_TOTAL_FEATS])[:,1]
    base["win_prob"], base["run_total_prob"] = wp, op
    base["win_conf"] = np.maximum(wp, 1-wp)
    base["run_total_conf"] = np.maximum(op, 1-op)

    # === 출력 ===
    SEP = "="*70
    print(SEP)
    print(f"  MLB 예측 — {TARGET_DATE}")
    print(f"  승패 선택: 확신 64%+ | 총득점 선택: 확신 60%+")
    print(SEP)
    win_selections, run_total_selections = 0, 0
    for _, g in base.iterrows():
        win_prediction = g["home_team"] if g["win_prob"]>=0.5 else g["away_team"]
        run_total_prediction = "UPPER" if g["run_total_prob"]>=0.5 else "LOWER"
        wb = "선택" if g["win_conf"]>=0.64 else ""
        ob = "선택" if g["run_total_conf"]>=0.60 else ""
        if wb: win_selections += 1
        if ob: run_total_selections += 1
        print(f"")
        print(f"  {g['away_team']} @ {g['home_team']}")
        print(f"    승패: {win_prediction} ({g['win_prob']:.1%}, 확신 {g['win_conf']:.1%}) {wb}")
        print(f"    RUN_TOTAL:  {run_total_prediction} {g['run_total_baseline']} ({g['run_total_prob']:.1%}, 확신 {g['run_total_conf']:.1%}) {ob}")
    print(f"")
    print(SEP)
    print(f"  선택 요약: 승패 {win_selections}경기, 총득점 {run_total_selections}경기")
    print(SEP)

    # predictions CSV 저장 (tracker용)
    pred_path = f"{DATA_2026}/predictions_{TARGET_DATE}.csv"
    base.to_csv(pred_path, index=False, encoding="utf-8-sig")
    print(f"\n  💾 저장: {pred_path}")

## 적중률 확인 (경기 종료 후)

In [ ]:
if "base" in dir() and "홈팀결과" in base.columns and base["홈팀결과"].notna().sum() > 0:
    br = base[base["홈팀결과"].notna()].copy()
    br["home_win"] = (br["홈팀결과"]=="승").astype(int)
    br["win_pred"] = (br["win_prob"]>=0.5).astype(int)
    br["total_runs"] = br["홈팀득점"] + br["원정팀득점"]
    br["run_total_pred"] = (br["run_total_prob"]>=0.5).astype(int)
    br["actual_upper"] = (br["total_runs"]>br["run_total_baseline"]).astype(int)
    push = br["total_runs"]==br["run_total_baseline"]

    wm = br["win_conf"]>=0.64
    om = (br["run_total_conf"]>=0.60) & (~push)

    if wm.sum() > 0:
        wa = (br.loc[wm,"win_pred"]==br.loc[wm,"home_win"]).mean()*100
        print(f"승패 선택: {wa:.0f}% ({wm.sum()}경기)")
        for _, g in br[wm].iterrows():
            ok = "✅" if g["win_pred"]==g["home_win"] else "❌"
            prediction = g["home_team"] if g["win_prob"]>=0.5 else g["away_team"]
            winner = g["home_team"] if g["home_win"]==1 else g["away_team"]
            print(f"  {ok} {prediction} → 실제 {winner} ({g['원정팀득점']:.0f}:{g['홈팀득점']:.0f})")

    if om.sum() > 0:
        oa = (br.loc[om,"run_total_pred"]==br.loc[om,"actual_upper"]).mean()*100
        print(f"")
        print(f"총득점 선택: {oa:.0f}% ({om.sum()}경기)")
        for _, g in br[om].iterrows():
            ok = "✅" if g["run_total_pred"]==g["actual_upper"] else "❌"
            prediction = "UPPER" if g["run_total_prob"]>=0.5 else "LOWER"
            actual = "UPPER" if g["actual_upper"]==1 else "LOWER"
            print(f"  {ok} {prediction} {g['run_total_baseline']} → 실제 {actual} ({g['total_runs']:.0f}점)")
else:
    print("⚠️ 경기결과 없음 — 결과 CSV 업데이트 후 다시 실행")